# Friedman Test

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon, norm

In [10]:
df_raw = pd.read_csv('data_CBR-RAG_2026-04-06_13-14_final.csv', encoding='utf-16', sep='\t')
df_raw = df_raw.copy()
df_raw['participant_id'] = df_raw['CASE']

DIMENSIONS = {
    '01': 'Helpfulness',
    '02': 'Clarity',
    '03': 'Naturalness',
    '04': 'Length Appropriateness',
    '05': 'Trustworthiness',
    '06': 'Process Grounding'
}

# Single patterns — one score each
PATTERNS_SINGLE = {
    'A0': {'Exact Match': 'Q101', 'Paraphrased': 'Q104', 'Pattern Adaptation': 'Q105'},
    'A1': {'Exact Match': 'Q123', 'Paraphrased': 'Q109', 'Pattern Adaptation': 'Q112'},
    'C':  {'Exact Match': 'Q137', 'Paraphrased': 'Q140', 'Pattern Adaptation': 'Q141'},
    'D':  {'Exact Match': 'Q144', 'Paraphrased': 'Q145', 'Pattern Adaptation': 'Q148'},
}

# Pattern B appears twice — B1 and B2 are averaged into one B score
PATTERN_B = {
    'B1': {'Exact Match': 'Q113', 'Paraphrased': 'Q116', 'Pattern Adaptation': 'Q117'},
    'B2': {'Exact Match': 'Q120', 'Paraphrased': 'Q121', 'Pattern Adaptation': 'Q136'},
}

CATEGORIES = ['Exact Match', 'Paraphrased', 'Pattern Adaptation']

print(f' {len(df_raw)} participants')
print(f'Single patterns: {list(PATTERNS_SINGLE.keys())}')
print(f'Pattern B (averaged): B1, B2')
print(f'Total effective patterns per category: 5 (A0, A1, B, C, D)')

 36 participants
Single patterns: ['A0', 'A1', 'C', 'D']
Pattern B (averaged): B1, B2
Total effective patterns per category: 5 (A0, A1, B, C, D)


Reverse Scores and Category Scores per Participant


In [11]:
records = []
for _, row in df_raw.iterrows():
    pid = row['participant_id']
    for cat in CATEGORIES:
        pattern_scores = []

        # A0, A1, C, D — one score each
        for pattern, mapping in PATTERNS_SINGLE.items():
            col = mapping[cat]
            dim_scores = [6 - row[f'{col}_{s}'] for s in DIMENSIONS if f'{col}_{s}' in df_raw.columns]
            if dim_scores:
                pattern_scores.append(np.mean(dim_scores))

        # Pattern B — average B1 and B2 into one score
        b_scores = []
        for pattern, mapping in PATTERN_B.items():
            col = mapping[cat]
            dim_scores = [6 - row[f'{col}_{s}'] for s in DIMENSIONS if f'{col}_{s}' in df_raw.columns]
            if dim_scores:
                b_scores.append(np.mean(dim_scores))
        if b_scores:
            pattern_scores.append(np.mean(b_scores))

        records.append({
            'participant_id': pid,
            'category':       cat,
            'score':          np.mean(pattern_scores)
        })

df_cat = pd.DataFrame(records)
print(f'Total rows: {len(df_cat)}')
df_cat.head(9)

Total rows: 108


,participant_id,category,score
0,115,Exact Match,3.583333
1,115,Paraphrased,3.316667
2,115,Pattern Adaptation,3.483333
3,121,Exact Match,3.900000
4,121,Paraphrased,3.466667
5,121,Pattern Adaptation,3.200000
6,128,Exact Match,4.366667
7,128,Paraphrased,4.400000
8,128,Pattern Adaptation,4.233333


## Descriptive Statistics per Category

In [12]:
desc = df_cat.groupby('category')['score'].agg(['mean','std','median']).round(3)
desc.columns = ['Mean', 'SD', 'Median']
desc = desc.reindex(CATEGORIES)

print('Descriptive statistics per category for CBR-RAG')
print('(1 = Strongly Disagree, 5 = Strongly Agree)')
print('=' * 50)
print(desc.to_string())

Descriptive statistics per category for CBR-RAG
(1 = Strongly Disagree, 5 = Strongly Agree)
                     Mean     SD  Median
category                                
Exact Match         3.951  0.390   4.000
Paraphrased         3.894  0.396   3.825
Pattern Adaptation  3.678  0.417   3.600


## Friedman Test


In [13]:
exact  = df_cat[df_cat['category'] == 'Exact Match']['score'].values
paraph = df_cat[df_cat['category'] == 'Paraphrased']['score'].values
adapt  = df_cat[df_cat['category'] == 'Pattern Adaptation']['score'].values

chi2, p_friedman = friedmanchisquare(exact, paraph, adapt)

n = len(exact)
k = 3
W = chi2 / (n * (k - 1))


print('Friedman Test')
print(f'chi2 statistic:  {chi2:.3f}')
print(f'p-value:         {p_friedman:.4f}')
print(f'df:              {k - 1}')
print(f'N:               {n}')
print()

Friedman Test
chi2 statistic:  17.133
p-value:         0.0002
df:              2
N:               36



In [14]:
df_wide = df_cat.pivot(index='participant_id', columns='category', values='score').reset_index()
df_wide.to_csv('category_scores_wide.csv', index=False)
print(df_wide.head())

category  participant_id  Exact Match  Paraphrased  Pattern Adaptation
0                    115     3.583333     3.316667            3.483333
1                    121     3.900000     3.466667            3.200000
2                    128     4.366667     4.400000            4.233333
3                    156     4.033333     4.083333            4.000000
4                    157     4.050000     3.816667            3.733333


## Post-Hoc Pairwise Comparisons (Wilcoxon Signed-Rank)

In [15]:
alpha_corrected = round(0.05 / 3, 4)

comparisons = [
    ('Exact Match',  exact,  'Paraphrased',        paraph),
    ('Paraphrased',  paraph, 'Pattern Adaptation', adapt),
    ('Exact Match',  exact,  'Pattern Adaptation', adapt),
]

print(f'Post-hoc Wilcoxon Signed-Rank Tests (one-sided)')
print('=' * 70)

posthoc_results = []
for (name_a, a, name_b, b) in comparisons:
    W_stat, p_val = wilcoxon(a, b, alternative='greater')
    p_adj = min(p_val * 3, 1.0)
    z = norm.ppf(1 - p_val)
    r = z / np.sqrt(n)

    if   p_adj < 0.001: sig = '***'
    elif p_adj < 0.01:  sig = '**'
    elif p_adj < 0.05:  sig = '*'
    else:               sig = 'ns'

    label = f'{name_a} > {name_b}'
    print(f'{label:<42}  W={W_stat:.1f}  p={p_val:.4f}  p_adj={p_adj:.4f}  r={r:.3f}  {sig}')

    posthoc_results.append({
        'Comparison':         label,
        'W':                  W_stat,
        'p':                  round(p_val, 4),
        'p_adj (Bonferroni)': round(p_adj, 4),
        'r':                  round(r, 3),
        'Sig.':               sig
    })

print()
print(f'Bonferroni corrected alpha = {alpha_corrected}')
print('*** p<0.001  ** p<0.01  * p<0.05  ns = not significant')
print('Effect size r: small>=0.1  medium>=0.3  large>=0.5')

Post-hoc Wilcoxon Signed-Rank Tests (one-sided)
Exact Match > Paraphrased                   W=383.5  p=0.1309  p_adj=0.3927  r=0.187  ns
Paraphrased > Pattern Adaptation            W=542.0  p=0.0005  p_adj=0.0015  r=0.547  **
Exact Match > Pattern Adaptation            W=577.5  p=0.0001  p_adj=0.0002  r=0.640  ***

Bonferroni corrected alpha = 0.0167
*** p<0.001  ** p<0.01  * p<0.05  ns = not significant
Effect size r: small>=0.1  medium>=0.3  large>=0.5


In [16]:
df_cat.to_csv('category_scores.csv', index=False)
pd.DataFrame(posthoc_results).to_csv('h3_posthoc_results.csv', index=False)